# Завдання: Створити RAG систему для аналізу фінансових звітів

**Мета** - Навчитися працювати з векторними базами даних, правильно розбивати текст на чанки та створювати ефективну систему пошуку інформації.



**Опис задачі** - Створіть систему, яка може відповідати на питання про фінансові показники компаній, використовуючи векторне сховище та LLM-моделі.

Система має правильно розбивати текст на логічні частини (по компаніях) та знаходити релевантну інформацію для відповіді.


**Рекомендований стек** - Chroma, OpenAIEmbeddings, RecursiveCharacterTextSplitter. Допускаються альтернативні інструменти, якщо RAG-логіка та якість відповідей зберігаються.

In [1]:
# Запускається 1 раз на початку сесії Colab
!pip install -q langchain langchain-community langchain-openai langchain-chroma chromadb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-cloud-bigquery 3.46.2 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
holidays 0.107 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requ

In [4]:
import os
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


api_key = userdata.get("OPENROUTER_API_KEY")

BASE_URL = "https://openrouter.ai/api/v1"
MODEL_NAME = "google/gemini-2.5-flash"   # недорога модель
EMBEDDING_MODEL = "text-embedding-3-small"
COLLECTION_NAME = "finansovi_zvity_kompaniy"


# ==========================================
# ВХІДНІ ДАНІ ТА ЧАНКУВАННЯ
# ==========================================
raw_text = """ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.

ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий прибуток 156 млн дол, маржа EBITDA 28 відсотків.
Працівників 5100 осіб. ROE становить 19.7 відсотка. Співвідношення борг до EBITDA дорівнює 0.9. Лідер у секторі відновлюваної енергії.

РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуток 48 млн дол, маржа EBITDA 12 відсотків.
Працівників 8500 осіб. ROE становить 9.3 відсотка. Співвідношення борг до EBITDA дорівнює 3.2. Мережа з 340 магазинів в Україні.

ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків.
Працівників 890 осіб. Від'ємний показник ROE. Стартап який активно інвестує у масштабування бізнесу."""

# Розділювач - порожній рядок між компаніями ("\n\n").
# Найдовший блок компанії ~260 символів, тому chunk_size=300 вміщує одну компанію,
# але не вміщує двох одразу (їх склеювання дало б >450 символів).
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"],
    chunk_size=300,
    chunk_overlap=0,
)

# Створюємо чанки
documents = text_splitter.create_documents([raw_text])

# Додаємо в metadata назву компанії - перевіряти, що саме знайшов retriever
for doc in documents:
    doc.metadata["company"] = doc.page_content.split(" за 2024 рік")[0]
    doc.metadata["year"] = 2024

print(f"Кількість створених чанків: {len(documents)}")
assert len(documents) == 4, "Очікується рівно 4 чанки - по одному на компанію"
for i, doc in enumerate(documents, 1):
    print(f"  Чанк {i}: {doc.metadata['company']} ({len(doc.page_content)} символів)")


# ==========================================
# ВЕКТОРНЕ СХОВИЩЕ ТА RETRIEVER
# ==========================================
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=api_key,
    openai_api_base=BASE_URL,
    check_embedding_ctx_length=False,  # надсилати чистий текст, а не токени
)

llm = ChatOpenAI(
    model=MODEL_NAME,
    api_key=api_key,
    base_url=BASE_URL,
    temperature=0,
)

# Якщо комірку запускають повторно в тій самій сесії, стара колекція лишається в пам'яті
# і документи задублюються (тоді k=2 може повернути 2 копії однієї компанії).
# Тому спочатку видаляємо стару колекцію.
try:
    Chroma(collection_name=COLLECTION_NAME, embedding_function=embeddings).delete_collection()
except Exception:
    pass

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    collection_metadata={"description": "Фінансові звіти українських компаній за 2024 рік"},
)
print("Документів у сховищі:", vectorstore._collection.count(), "(має бути 4)")

# Retriever: 2 найрелевантніші чанки (k=2)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# ==========================================
# СТВОРЕННЯ RAG ЛАНЦЮГА
# ==========================================
prompt = ChatPromptTemplate.from_template(
    """Ти - фінансовий аналітик. Відповідай на запитання користувача українською мовою
СТРОГО та ВИКЛЮЧНО на основі наданого контексту.
Наводь конкретні цифри з контексту разом з одиницями виміру (млн дол, відсотки, осіб тощо).
Якщо в контексті немає відповіді або запитання стосується компанії, якої немає в контексті,
чітко напиши, що інформація про це відсутня в наданому контексті. Нічого не вигадуй.

Контекст:
{context}

Запитання: {question}
Відповідь:"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


# ==========================================
# ТЕСТОВІ ПИТАННЯ
# ==========================================
questions = [
    "Яка виручка у компанії ТехНова Інк за 2024 рік?",
    "Скільки працівників у ГрінЕнерджі Солюшнс?",
    "Який показник ROE у РітейлМакс Груп?",
    "Де знаходиться штаб-квартира ТехНова Інк?",
    "Яка маржа EBITDA у ФінТех Дайнамікс?",
    "Який чистий прибуток у ЕкоПакування Солюшнс?"
]

print("\n" + "="*70)

for i, q in enumerate(questions, 1):
    answer = rag_chain.invoke(q)
    print(f"\nПитання {i}: {q}")
    print(f"Відповідь: {answer}")
    print("-" * 70)

Кількість створених чанків: 4
  Чанк 1: ТехНова Інк (235 символів)
  Чанк 2: ГрінЕнерджі Солюшнс (260 символів)
  Чанк 3: РітейлМакс Груп (249 символів)
  Чанк 4: ФінТех Дайнамікс (230 символів)
Документів у сховищі: 4 (має бути 4)


Питання 1: Яка виручка у компанії ТехНова Інк за 2024 рік?
Відповідь: Виручка у компанії ТехНова Інк за 2024 рік становить 450 млн дол.
----------------------------------------------------------------------

Питання 2: Скільки працівників у ГрінЕнерджі Солюшнс?
Відповідь: У ГрінЕнерджі Солюшнс 5100 осіб працівників.
----------------------------------------------------------------------

Питання 3: Який показник ROE у РітейлМакс Груп?
Відповідь: ROE РітейлМакс Груп становить 9.3 відсотка.
----------------------------------------------------------------------

Питання 4: Де знаходиться штаб-квартира ТехНова Інк?
Відповідь: Штаб-квартира ТехНова Інк знаходиться у Львові.
----------------------------------------------------------------------

Питання 5: Яка ма